<a href="https://colab.research.google.com/github/avivbrook/modular-nested-exponentiation/blob/main/modular-nested-exponentiation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Summary

We present an algorithm that takes as input an arbitrarily long sequence of positive integers $a_1,a_2,\ldots,a_\ell$ and a positive integer $m$ and computes
$$a_1^{a_2^{\cdot^{\cdot^{a_\ell}}}}\bmod m$$
efficiently (that is, without computing the value of the nested exponent).

# Notation

For convenience, we define an operator $E$ to serve as a shorthand for describing nested exponentiation.

---

> **Definition (nested exponentiation operator).** Given a tuple of $\ell$ numbers $(a_1,a_2,\ldots,a_\ell)$, define the operator $E$ recursively as follows.
$$E(a_1,a_2,\ldots,a_\ell)=\begin{cases}
    1&\text{if $\ell=0$,}\\
    a_1^{E(a_2,\ldots,a_\ell)}&\text{if $\ell\gt0$.}
\end{cases}$$
We call $a_1$ the **base** and $E(a_2,\ldots,a_\ell)$ the **exponent** of $E(a_1,a_2,\ldots,a_\ell)$. We have that
$$E(a_1,a_2,\ldots,a_\ell)=a_1^{a_2^{\cdot^{\cdot^{a_\ell}}}}\bmod m\text{.}$$

---

We are therefore interested in computing $E(a_1,a_2,\ldots,a_\ell)\bmod m$.

# Preliminaries

`BaseTest` is a subclass of `TestCase` from the `unittest` library. We use it to test and benchmark all the functions implemented in this notebook.

In [1]:
from typing import List, Callable, Tuple, Dict
from decimal import Decimal
from math import ceil, gcd
from sympy import primerange
from sympy.ntheory import isprime, totient, factorint
from unittest import TestCase, TextTestRunner, TestSuite
from time import time
from numpy import format_float_scientific, format_float_positional
from random import randint, sample, randrange
from enum import IntFlag, auto
from array import array
from collections import Counter
from statistics import mean

class BaseTest(TestCase):
    MAX = 2**63
    CASES = [3, 1000]

    class Col:
        """Represents a column in a table."""
        L, R = False, True # left, right align
        def __init__(self, head: str, data: List[str], align: bool = L):
            self.head, self.data, self.align = head, data, align

    class V(IntFlag):
        """Verbosity flags."""
        R = auto() # runs
        SB = auto() # basic stats
        SE = auto() # extra stats
        S = SB | SE # all stats
        A = R | S # all

    @staticmethod
    def print_table(cols: List[Col]):
        pad, fmt = list(), list() # padding, format string
        for col in cols:
            if len(col.data) != len(cols[0].data):
                raise ValueError('all columns must have equal length')
            p = max(len(col.head), len(max(col.data, key=len)))
            pad.append(p)
            if col.align == col.L:
                fmt.append('%%-%is'%p)
            else:
                fmt.append('%%%is'%p)
        print(*(fmt[i]%col.head for i,col in enumerate(cols)), sep=' | ')
        print(*(fmt[i]%('-'*(pad[i]+1)) for i in range(len(pad))), sep='|-')
        for j in range(len(cols[0].data)):
            print(*(fmt[i]%cols[i].data[j] for i in range(len(cols))),
                  sep=' | ')
    
    @staticmethod
    def fpos(x: float) -> str:
        return format_float_positional(x, trim='-', precision=2)
    
    @staticmethod
    def fsci(x: float) -> str:
        if x:
            return format_float_scientific(x, precision=2, trim='-',
                                           exp_digits=1).replace('e', 'E')
        return BaseTest.fpos(x)

    def compare_fns(self, fns: List[Callable], cases: List[Tuple],
                    val: bool = False, val_set: list = None,
                    val_fn: Callable = None, verb: V = V(0),
                    name_fn: Callable = None):
        avg_times = {fn: 0 for fn in fns}
        win_rates = {fn: 0 for fn in fns}
        deltas = {fn: 0 for fn in fns}
        delta_pers = {fn: 0 for fn in fns}
        num_cases = len(cases)
        for i, case in enumerate(cases):
            results, times = list(), dict()
            if self.V.R in verb:
                if name_fn:
                    print('%s = '%name_fn(*case), end='')
                else:
                    print('%s('%fns[0].__name__, end='')
                    print(*case, sep=', ', end=') = ')
            for fn in fns:
                t0 = time()
                res = fn(*case)
                t1 = time()
                delta_t = t1-t0
                times[fn] = delta_t
                results.append(res)
                if val:
                    if val_set:
                        self.assertEqual(res, val_set[i])
                    elif val_fn:
                        self.assertTrue(val_fn(case, res))
                    else:
                        self.assertEqual(res, results[0])
                if self.V.R in verb:
                    if len(times) == 1:
                        print(res)
                    print('\t%s took %ss'%(fn.__name__, self.fsci(delta_t)))
            winner = min(times.keys(), key=lambda fn: times[fn])
            win_t = times[winner]
            if win_t:
                win_rates[winner] += 1
                for fn in fns:
                    avg_times[fn] += times[fn]
                    deltas[fn] += times[fn]-win_t
                    delta_pers[fn] += (times[fn]-win_t)/win_t
            else:
                num_cases -= 1
            if self.V.R in verb:
                print('-> winner:', winner.__name__)
        if self.V.SB in verb:
            for fn in fns:
                avg_times[fn] /= num_cases
                win_rates[fn] *= 100/num_cases
                deltas[fn] /= num_cases
                delta_pers[fn] *= 100/num_cases
            if self.V.R in verb:
                print()
            print('Statistics from %i test cases:'%num_cases)
            fns.sort(key=lambda fn: win_rates[fn], reverse=True)
            cols = [self.Col('STATS', ['#%i.'%(i+1) for i in range(len(fns))]),
                    self.Col('Function', [fn.__name__ for fn in fns]),
                    self.Col('Mean time',
                             ['%ss'%self.fsci(avg_times[fn]) for fn in fns],
                             self.Col.R)]
            if self.V.SE in verb:
                cols.extend([self.Col('Win %',
                                      ['%s%%'%self.fpos(win_rates[fn])
                                      for fn in fns]),
                             self.Col('Avg Δtime from optimal',
                                      ['%ss'%self.fsci(deltas[fn])
                                      for fn in fns],
                                      self.Col.R),
                             self.Col('Avg %Δtime from optimal',
                                      ['%s%%'%self.fpos(delta_pers[fn])
                                      for fn in fns], self.Col.R)])
            self.print_table(cols)
        if verb:
            print()

    aux = None

    def test(self, n: Dict[int, int] = {CASES[0]: V.R, CASES[1]: V.S}):
        for num_cases, verb in n.items():
            self.aux(num_cases, verb)
    
    def run_test(self):
        TextTestRunner(verbosity=0).run(TestSuite([self.__class__('test')]))

## Simple modular exponentiation

`mod_exp` takes as input positive integers $b,e,m$ and returns $b^e\bmod m$.

`mod_exp(b,e,m)` is equivalent to `pow(b,e,m)`.

It is unlikely that our implementation will be very efficient since `pow` is implemented in C.

In [2]:
def mod_exp(b: int, e: int, m: int) -> int:
    """
    Given an integer b, a nonnegative integer e, and a positive integer m,
    computes bᵉ mod m using binary exponentiation.

    Parameters
    ----------
    b: integer
        The base.
    e: nonnegative integer
        The exponent.
    m: positive integer
        The modulus of congruence.
    
    Returns
    -------
        bᵉ mod m.
    """
    if m < 1:
        raise ValueError('m must be a positive integer')
    if e < 0:
        raise ValueError('e must be a nonnegative integer')
    if m == 1:
        return 0
    
    b %= m
    result = 1
    while e: # while e > 0
        if e&1: # if e%2 == 1
            result = result*b%m
        e >>= 1
        b = b*b%m
    return result

class TestModExp(BaseTest):
    def aux(self, num_cases: int, verb: BaseTest.V):
        builtin = lambda b,e,m: pow(b, e, m)
        builtin.__name__ = 'builtin pow'
        custom = lambda b,e,m: mod_exp(b, e, m)
        custom.__name__ = 'our mod_exp'
        cases = list()
        for _ in range(num_cases):
            cases.append((randrange(-self.MAX, self.MAX),
                          randrange(self.MAX),
                          randrange(1, self.MAX)))
        self.compare_fns([builtin, custom], cases, val=True, verb=verb,
                         name_fn=lambda b,e,m: 'E(%i, %i) mod %i'%(b,e,m))

if __name__ == '__main__':
    TestModExp().run_test()

E(-3035747851080014667, 8466667564359533083) mod 8208746567837582071 = 2768001418113446517
	builtin pow took 2.96E-5s
	our mod_exp took 4.74E-5s
-> winner: builtin pow
E(-1325894918320583586, 287146590633090928) mod 3191385631042883643 = 842304168785062911
	builtin pow took 4.01E-5s
	our mod_exp took 4.63E-5s
-> winner: builtin pow
E(-3421005079099468685, 8959160348520878556) mod 3678103831363059540 = 489095224071287185
	builtin pow took 3.12E-5s
	our mod_exp took 6.39E-5s
-> winner: builtin pow

Statistics from 1000 test cases:
STATS | Function    | Mean time | Win % | Avg Δtime from optimal | Avg %Δtime from optimal
------|-------------|-----------|-------|------------------------|-------------------------
#1.   | builtin pow |  2.03E-5s | 98.5% |               5.98E-7s |                   1.75%
#2.   | our mod_exp |  3.16E-5s | 1.5%  |               1.19E-5s |                  61.29%



----------------------------------------------------------------------
Ran 1 test in 0.066s

OK


### Results

Python's built-in `pow` function outperforms `mod_exp`. We are within the same order of magnitude, but `pow` is the clear winner.

## The Euclidean algorithm

We implement a function `gcd_euclid` that takes as input integers $a$ and $b$ and returns $\gcd(a,b)$.

`gcd_euclid(a,b)` is equivalent to `math.gcd(a,b)`.

It is once again unlikely that our implementation will be very efficient when compared to `math.gcd` since the `math` library implements it in C, whereas we implement it in pure Python.

In [3]:
def gcd_euclid(a: int, b: int) -> int:
    """
    Given nonnegative integers a and b, computes gcd(a,b) using the Euclidean
    algorithm.

    Parameters
    ----------
    a, b: nonnegative integers.

    Returns
    -------
        gcd(a,b).
    """
    if a < 0 or b < 0:
        raise ValueError('a and b must be nonnegative integers')
    
    while a: # while remainder > 0
        a, b = b%a, a
    return b # last nonzero remainder

class TestGCD(BaseTest):
    def aux(self, num_cases: int, verb: BaseTest.V):
        lib = lambda a,b: gcd(a, b)
        lib.__name__ = 'math.gcd'
        custom = lambda a,b: gcd_euclid(a, b)
        custom.__name__ = 'our gcd_euclid'
        cases = list()
        for _ in range(num_cases):
            cases.append((randrange(self.MAX), randrange(self.MAX)))
        self.compare_fns([lib, custom], cases, val=True, verb=verb,
                         name_fn=lambda a,b: 'gcd(%i, %i)'%(a,b))

if __name__ == '__main__':
    TestGCD().run_test()

gcd(2556962555604902324, 4836440615798425723) = 1
	math.gcd took 3.81E-6s
	our gcd_euclid took 1.03E-5s
-> winner: math.gcd
gcd(8193915206524912662, 1128667354622349918) = 6
	math.gcd took 3.34E-6s
	our gcd_euclid took 9.78E-6s
-> winner: math.gcd
gcd(6801280051501449134, 6844428010946011304) = 2
	math.gcd took 2.62E-6s
	our gcd_euclid took 7.87E-6s
-> winner: math.gcd

Statistics from 1000 test cases:
STATS | Function       | Mean time | Win % | Avg Δtime from optimal | Avg %Δtime from optimal
------|----------------|-----------|-------|------------------------|-------------------------
#1.   | math.gcd       |  1.26E-6s | 99.8% |               1.18E-7s |                   2.09%
#2.   | our gcd_euclid |  3.76E-6s | 0.2%  |               2.62E-6s |                 238.18%



----------------------------------------------------------------------
Ran 1 test in 0.018s

OK


### Results

`math.gcd` outperforms our `gcd_euclid` function significantly.

## The extended Euclidean algorithm

`ext_gcd` that takes as input integers $a$ and $b$ and returns $\gcd(a,b)$ as well as integers $x$ and $y$ such that
$$ax+by=\gcd(a,b)\text{.}$$

In [4]:
def ext_gcd(a: int, b: int) -> Tuple[int, int, int]:
    """
    Given nonnegative integers a and b, uses the extended Euclidean algorithm to
    compute gcd(a,b) and the coefficients of Bézout's identity -- integers x and
    y such that
        ax + by = gcd(a,b).
    
    Inputs are not validated. That is, a and b are assumed to be nonnegative
    integers.
    
    Parameters
    ----------
    a, b: nonnegative integers.

    Returns
    -------
        gcd(a,b), x, y.
    """
    if a < 0 or b < 0:
        raise ValueError('a and b must be nonnegative integers')
    
    prev_r, r, prev_x, x = a, b, 1, 0
    while r: # while remainder > 0
        q = prev_r // r # quotient
        prev_r, prev_x, r, x = r, x, prev_r-q*r, prev_x-q*x
    return prev_r, prev_x, ((prev_r - prev_x*a)//b if b else 0)

class TestExtGCD(BaseTest):    
    def test(self):
        lib = lambda a,b: gcd(a, b)
        lib.__name__ = 'math.gcd'
        custom = lambda a,b: ext_gcd(a, b)
        custom.__name__ = 'our ext_gcd'
        cases = list()
        for _ in range(self.CASES[0]):
            cases.append((randrange(self.MAX), randrange(self.MAX)))
        self.compare_fns([custom], cases, val=True,
                         val_fn=lambda c,r: r[0]==gcd(c[0],c[1])==
                                            c[0]*r[1]+c[1]*r[2], verb=self.V.R,
                         name_fn=lambda a,b: 'ext_gcd(%i, %i)'%(a,b))
        cases = list()
        for _ in range(self.CASES[1]):
            cases.append((randrange(self.MAX), randrange(self.MAX)))
        self.compare_fns([lib, custom], cases, verb=self.V.S,
                         name_fn=lambda a,b: 'gcd(%i, %i)'%(a,b))

if __name__ == '__main__':
    TestExtGCD().run_test()

ext_gcd(7698530593486688715, 2213720584515450086) = (1, -1015789404662473401, 3532553233246182206)
	our ext_gcd took 2.74E-5s
-> winner: our ext_gcd
ext_gcd(2683607031991701867, 2320036719720796941) = (3, 271930031469469465, -314543877024935472)
	our ext_gcd took 2.17E-5s
-> winner: our ext_gcd
ext_gcd(7300182470718120712, 4158796045662065328) = (8, 130407973583594633, -228912885446699996)
	our ext_gcd took 2.1E-5s
-> winner: our ext_gcd

Statistics from 1000 test cases:
STATS | Function    | Mean time | Win % | Avg Δtime from optimal | Avg %Δtime from optimal
------|-------------|-----------|-------|------------------------|-------------------------
#1.   | math.gcd    |  1.22E-6s | 100%  |                     0s |                      0%
#2.   | our ext_gcd |  1.31E-5s | 0%    |               1.19E-5s |                 980.91%



----------------------------------------------------------------------
Ran 1 test in 0.025s

OK


### Results

`math.gcd` outperforms our `ext_gcd` function which is not surprising since `ext_gcd` performs more operations.

## Prime number sieve

`prime_sieve` takes as input an integer $n$ and returns a list of all primes $\lt n$.

In [5]:
def prime_sieve(n: int) -> array:
    """
    Given an integer n, finds all primes < n using the sieve of Robert William
    Hanks from StackOverflow: https://stackoverflow.com/a/3035188.

    Parameters
    ----------
    n: integer.

    Returns
    -------
        array of all primes < n.
    """
    if n < 3:
        return array('l')
    elif n == 3:
        return array('l', [2])
    elif n < 6:
        return array('l', [2, 3])
    
    n, correction = n-n%6+6, 2-(n%6>1)
    sieve = [True] * (n//3)
    for i in range(1, int(n**0.5)//3+1):
        if sieve[i]:
            k = 3*i+1 | 1
            sieve[k*k//3::2*k] = [False]*((n//6-k*k//6-1)//k+1)
            sieve[k*(k-2*(i&1)+4)//3::2*k] = [False]*((n//6-k*
                                                       (k-2*(i&1)+4)//6-1)//k+1)
    return array('l', [2, 3]+[3*i+1|1 for i in range(1, n//3-correction)
                              if sieve[i]])

max_prime = 1000
small_primes = prime_sieve(max_prime)

class TestSieve(BaseTest):
    def test(self):
        lib = lambda n: array('l', primerange(1,n))
        lib.__name__ = 'sympy.primerange'
        custom = lambda n: prime_sieve(n)
        custom.__name__ = 'our prime_sieve'
        cases = list()
        for _ in range(self.CASES[0]):
            cases.append((randrange(100),))
        self.compare_fns([lib, custom], cases, val=True,
                         verb=self.V.R, name_fn=lambda n: 'sieve(%i)'%n)
        cases = list()
        for _ in range(10):
            cases.append((randrange(100000),))
        self.compare_fns([lib, custom], cases, val=True,
                         verb=self.V.S, name_fn=lambda n: 'sieve(%i)'%n)

if __name__ == '__main__':
    TestSieve().run_test()

sieve(72) = array('l', [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71])
	sympy.primerange took 4.92E-4s
	our prime_sieve took 2.41E-5s
-> winner: our prime_sieve
sieve(2) = array('l')
	sympy.primerange took 9.78E-5s
	our prime_sieve took 2.38E-6s
-> winner: our prime_sieve
sieve(30) = array('l', [2, 3, 5, 7, 11, 13, 17, 19, 23, 29])
	sympy.primerange took 1.14E-4s
	our prime_sieve took 1.86E-5s
-> winner: our prime_sieve

Statistics from 10 test cases:
STATS | Function         | Mean time | Win % | Avg Δtime from optimal | Avg %Δtime from optimal
------|------------------|-----------|-------|------------------------|-------------------------
#1.   | our prime_sieve  |  2.19E-3s | 100%  |                     0s |                      0%
#2.   | sympy.primerange |  6.92E-2s | 0%    |                6.7E-2s |                2718.65%



----------------------------------------------------------------------
Ran 1 test in 0.723s

OK


### Results

`prime_sieve` beats `sympy`'s `primerange`.

## Primality test

`is_prime` takes as input an integer $n$ and returns `True` iff it is likely to be prime.

`is_prime(n)` is equivalent to `sympy.ntheory.isprime(n)`.

In [6]:
def is_prime(n: int, k: int = 3) -> bool:
    """
    Given an integer n and a positive integer k, determines if n is likely to be
    prime using the Miller-Rabin primality test. The probability of a false
    positive is 1/4ᵏ.

    Parameters
    ----------
    n: integer.
    k: positive integer, optional (default is 3)
        The number of rounds of testing to perform.
    
    Returns
    -------
        True if n is likely to be prime; False otherwise.
    """
    if k < 1:
        raise ValueError('k must be a positive integer')
    if n < 2:
        return False
    elif n < 4:
        return n > 1 # 2 and 3 are trivial primes
    elif not n&1: # if n%2 == 0
        return False
    elif n < max_prime:
        return n in small_primes
    
    d, r = n-1, 0 # n = 2ʳ·d + 1
    while not d&1: # while d%2 == 0
        d >>= 1
        r += 1
    
    for _ in range(k):
        x = pow(randrange(2, n-1), d, n)
        if x == 1 or x == n-1:
            continue
        for _ in range(r-1):
            x = pow(x, 2, n)
            if x == 1:
                return False
            elif x == n-1:
                break
        else:
            return False
    return True

class TestIsPrime(BaseTest):
    def aux(self, num_cases: int, verb: BaseTest.V):
        lib = lambda n: isprime(n)
        lib.__name__ = 'sympy.ntheory.isprime'
        custom = lambda n: is_prime(n)
        custom.__name__ = 'our is_prime'
        cases = list()
        for _ in range(num_cases):
            cases.append((randrange(self.MAX),))
        self.compare_fns([lib, custom], cases, val=True, verb=verb,
                         name_fn=lambda n: 'isprime(%i)'%n)

if __name__ == '__main__':
    TestIsPrime().run_test()

isprime(3259392011406130945) = False
	sympy.ntheory.isprime took 1.03E-5s
	our is_prime took 5.32E-5s
-> winner: sympy.ntheory.isprime
isprime(5045113620416250780) = False
	sympy.ntheory.isprime took 3.81E-6s
	our is_prime took 1.19E-6s
-> winner: our is_prime
isprime(7306344602886204293) = False
	sympy.ntheory.isprime took 3.34E-6s
	our is_prime took 4.2E-5s
-> winner: sympy.ntheory.isprime

Statistics from 1000 test cases:
STATS | Function              | Mean time | Win % | Avg Δtime from optimal | Avg %Δtime from optimal
------|-----------------------|-----------|-------|------------------------|-------------------------
#1.   | our is_prime          |  1.48E-5s | 62.6% |               8.63E-6s |                 613.49%
#2.   | sympy.ntheory.isprime |  9.91E-6s | 37.4% |               3.75E-6s |                   80.2%



----------------------------------------------------------------------
Ran 1 test in 0.036s

OK


### Results

Our `is_prime` function beats `sympy.ntheory.isprime` most of the time but it has much higher variance and overall slower average time. We will therefore still use `sympy.ntheory.isprime` as our primality test.

## Integer factorisation

`prime_factors` takes as input a positive integer $n$ and returns a a dict containing the prime factors of $n$ as keys and their respective multiplicities as values.

`prime_factors(n)` is equivalent to `sympy.ntheory.factorint(n)`.

In [7]:
def pollard_brent(n: int) -> int:
    """
    Given a positive integer n, returns a nontrivial factor of n if n is
    composite; if n is prime, returns n. Uses Pollard's rho algorithm with
    Brent's cycle finding method:
    https://comeoncodeon.wordpress.com/2010/09/18/pollard-rho-brent-integer-factorization/.

    Parameters
    ----------
    n: positive integer.

    Returns
    -------
        a nontrivial factor of n if n is composite; n otherwise.
    """
    if not n&1: # if n%2 == 0
        return 2
    elif not n%3: # if n%3 == 0
        return 3
    
    y, c, m = randrange(1, n), randrange(1, n), randrange(1, n)
    g, r, q = 1, 1, 1
    while g == 1:
        x = y
        for i in range(r):
            y = (pow(y, 2, n) + c) % n
        k = 0
        while k < r and g == 1:
            ys = y
            for i in range(min(m, r-k)):
                y = (pow(y, 2, n) + c) % n
                q = q*abs(x-y) % n
            g = gcd(q, n)
            k += m
        r *= 2
    if g == n:
        while True:
            ys = (pow(ys, 2, n) + c) % n
            g = gcd(abs(x-ys), n)
            if g > 1:
                break
    return g

def large_factors(n: int) -> Dict[int, int]:
    """
    Given a positive integer n, returns a dict containing the prime factors of n
    as keys and their respective multiplicities as values.

    Parameters
    ----------
    n: positive integer.

    Returns
    -------
        a dict containing the prime factors of n as keys and their respective
        multiplicities as values.
    """
    factors = Counter()
    while n > 1:
        if isprime(n):
            factors[n] += 1
            break
        factor = pollard_brent(n)
        factors += large_factors(factor)
        n //= factor
    return factors

def prime_factors(n: int) -> Dict[int, int]:
    """
    Given a positive integer n, returns a dict containing the prime factors of n
    as keys and their respective multiplicities as values.

    Parameters
    ----------
    n: positive integer.

    Returns
    -------
        a dict containing the prime factors of n as keys and their respective
        multiplicities as values.
    """
    if n < 1:
        raise ValueError('n must be a positive integer')

    factors = Counter()
    for p in small_primes:
        while not n%p: # while n%p == 0
            factors[p] += 1
            n //= p
    if n == 1:
        return factors
    return factors + large_factors(n)

class TestFactorInt(BaseTest):
    def aux(self, num_cases: int, verb: BaseTest.V):
        lib = lambda n: factorint(n)
        lib.__name__ = 'sympy.ntheory.factorint'
        custom = lambda n: prime_factors(n)
        custom.__name__ = 'our prime_factors'
        cases = list()
        for _ in range(num_cases):
            cases.append((randrange(self.MAX),))
        self.compare_fns([lib, custom], cases, val=True, verb=verb,
                         name_fn=lambda n: 'factorint(%i)'%n)

if __name__ == '__main__':
    TestFactorInt().run_test()

factorint(7188006746931883289) = {641: 1, 19213: 1, 583653727933: 1}
	sympy.ntheory.factorint took 4.07E-3s
	our prime_factors took 4.19E-4s
-> winner: our prime_factors
factorint(8077196671816748869) = {1577839: 1, 42384449: 1, 120779: 1}
	sympy.ntheory.factorint took 3.24E-2s
	our prime_factors took 7.1E-3s
-> winner: our prime_factors
factorint(8344920216934717036) = {2: 2, 5407: 1, 385838737605637: 1}
	sympy.ntheory.factorint took 1.91E-3s
	our prime_factors took 4.9E-4s
-> winner: our prime_factors

Statistics from 1000 test cases:
STATS | Function                | Mean time | Win % | Avg Δtime from optimal | Avg %Δtime from optimal
------|-------------------------|-----------|-------|------------------------|-------------------------
#1.   | our prime_factors       |  2.91E-3s | 89.4% |               7.35E-4s |                  35.53%
#2.   | sympy.ntheory.factorint |  1.01E-2s | 10.6% |               7.95E-3s |                 213.53%



----------------------------------------------------------------------
Ran 1 test in 13.104s

OK


### Results

Our `prime_factors` function beats `sympy.ntheory.factorint` quite consistently.

## Euler's totient function

`euler_totient` takes as input a positive integer $n$ and returns $\varphi(n)$, the number of positive integers $\leq n$ that are coprime to $n$.

`euler_totient(n)` is equivalent to `sympy.ntheory.totient(n)`.

In [8]:
def euler_totient(n: int) -> int:
    """
    Given a positive integer n, returns φ(n), which is the number of positive
    integers ≤ n that are coprime to n. Uses the product formula for φ: letting
        n = p₁ᵉ¹p₂ᵉ²···pᵣᵉʳ
    be the unique prime factorisation of n, where r is a nonnegative integer,
    p₁ < p₂ < ··· < pᵣ are distinct primes, and each eᵢ is a positive integer,
        φ(n) = n·(1 - 1/p₁)·(1 - 1/p₂)···(1 - 1/pᵣ).

    Parameters
    ----------
    n: positive integer.

    Returns
    -------
        φ(n).
    """
    if n < 1:
        raise ValueError('n must be a positive integer')
    
    for p in prime_factors(n).keys():
        n -= n//p
    return n

class TestTotient(BaseTest):
    def aux(self, num_cases: int, verb: BaseTest.V):
        lib = lambda n: totient(n)
        lib.__name__ = 'sympy.ntheory.totient'
        custom = lambda n: euler_totient(n)
        custom.__name__ = 'our euler_totient'
        cases = list()
        for _ in range(num_cases):
            cases.append((randrange(self.MAX),))
        self.compare_fns([lib, custom], cases, val=True, verb=verb,
                         name_fn=lambda n: 'φ(%i)'%n)

if __name__ == '__main__':
    TestTotient().run_test()

φ(7933738249149980319) = 4525380530253273600
	sympy.ntheory.totient took 2.81E-3s
	our euler_totient took 1.87E-3s
-> winner: our euler_totient
φ(6168199646776057593) = 4044719109444713280
	sympy.ntheory.totient took 3.24E-3s
	our euler_totient took 2.27E-3s
-> winner: our euler_totient
φ(5522630106273049275) = 2675972604905776000
	sympy.ntheory.totient took 6.43E-4s
	our euler_totient took 2.52E-4s
-> winner: our euler_totient

Statistics from 1000 test cases:
STATS | Function              | Mean time | Win % | Avg Δtime from optimal | Avg %Δtime from optimal
------|-----------------------|-----------|-------|------------------------|-------------------------
#1.   | our euler_totient     |  2.62E-3s | 90.3% |               4.85E-4s |                  25.34%
#2.   | sympy.ntheory.totient |  8.37E-3s | 9.7%  |               6.23E-3s |                 254.03%



----------------------------------------------------------------------
Ran 1 test in 11.040s

OK


### Results

Our `euler_totient` function beats `sympy.ntheory.totient`. This makes sense since this function's runtime relies almost solely on the time it takes to factor the input integer. Since our `prime_factors` is more efficient than `sympy`'s, one would expect our totient function to also be more efficient.

# The algorithm

## `pow_lt`

`pow_lt` takes as input a sequence of positive integers $e_1,e_2,\ldots,e_\ell$ and a positive number $k$ and returns `True` iff $E(e_1,e_2,\ldots,e_\ell)\lt k$.

In [9]:
def pow_lt_naive(seq: List[Decimal], k: Decimal) -> bool:
    """
    Given an arbitrarily long sequence of positive numbers
    seq = [e₁, e₂, e₃, ...] and a positive number k, returns True if and only if
    E(e₁, e₂, e₃, ...) < k.

    Inputs are not validated. That is, k and the elements of seq are assumed to
    be positive.

    Parameters
    ----------
    seq: list of positive numbers.
    k: positive number.

    Returns
    -------
        True if and only if E(seq[0], seq[1], seq[2], ...) < k.
    """
    if not len(seq): # if len(seq) == 0
        return 1 < k
    
    def _pow_lt(seq, k):
        """Auxiliary function."""
        if len(seq) == 1 or seq[0] == 1:
            return seq[0] < k
        l = Decimal(k).ln()/Decimal(seq[0]).ln() # high precision logarithm
        return _pow_lt(seq[1:], l) if l > 1 else False

    return _pow_lt(seq, k)

def pow_lt(seq: List[Decimal], k: Decimal) -> bool:
    """
    Given an arbitrarily long sequence of positive integers
    seq = [e₁, e₂, e₃, ...] and a positive number k, returns True if and only if
    E(e₁, e₂, e₃, ...) < k.

    Inputs are not validated. That is, k is assumed to be positive and the
    elements of seq are assumed to be positive integers.

    Parameters
    ----------
    seq: list of positive integers.
    k: positive number.

    Returns
    -------
        True if and only if E(seq[0], seq[1], seq[2], ...) < k.
    """
    if not len(seq): # if len(seq) == 0
        return 1 < k

    def _pow_lt(seq, k):
        """Auxiliary function."""
        if len(seq) == 1 or seq[0] == 1:
            return seq[0] < k
        if seq[1]*(seq[0].bit_length()-1) >= ceil(k).bit_length():
            return False
        l = Decimal(k).ln()/Decimal(seq[0]).ln() # high precision logarithm
        return _pow_lt(seq[1:], l) if l > 1 else False

    return _pow_lt(seq, k)

class TestPowLt(BaseTest):
    def test(self):
        naive = lambda seq,k: pow_lt_naive(seq, k)
        naive.__name__ = 'pow_lt_naive'
        efficient = lambda seq,k: pow_lt(seq, k)
        efficient.__name__ = 'pow_lt'
        cases = [([2,3,4], 2417851639229258349412352),
                 ([2,3,4], 2417851639229258349412353),
                 ([2,3,4], 2417851639229258349412351)] # some edge cases
        val_set = [False, True, False]
        self.compare_fns([naive, efficient], cases, val=True, val_set=val_set,
                         verb=self.V.R,
                         name_fn=lambda seq,k: 'E(%s) < %i'%
                         (str(seq).strip('[]'),k))
        cases = [(list(), randrange(1, self.MAX))]
        for _ in range(self.CASES[0]):
            seq = list()
            for _ in range(randint(1,10)):
                seq.append(randrange(1, self.MAX))
            cases.append((seq, randrange(1, self.MAX)))
        self.compare_fns([naive, efficient], cases, val=True, verb=self.V.R,
                         name_fn=lambda seq,k: 'E(%s) < %i'%
                         (str(seq).strip('[]'),k))
        cases = list()
        for _ in range(self.CASES[1]):
            seq = list()
            for _ in range(randint(1,10)):
                seq.append(randrange(1,self.MAX))
            cases.append((seq, randrange(1, self.MAX)))
        self.compare_fns([naive, efficient], cases, val=True, verb=self.V.S,
                         name_fn=lambda seq,k: 'E(%s) < %i'%
                         (str(seq).strip('[]'),k))
        
if __name__ == '__main__':
    TestPowLt().run_test()

E(2, 3, 4) < 2417851639229258349412352 = False
	pow_lt_naive took 2.26E-4s
	pow_lt took 2.16E-4s
-> winner: pow_lt
E(2, 3, 4) < 2417851639229258349412353 = True
	pow_lt_naive took 2.23E-4s
	pow_lt took 2.01E-4s
-> winner: pow_lt
E(2, 3, 4) < 2417851639229258349412351 = False
	pow_lt_naive took 1.97E-4s
	pow_lt took 2.1E-4s
-> winner: pow_lt_naive

E() < 2301396180883134089 = True
	pow_lt_naive took 1.43E-6s
	pow_lt took 9.54E-7s
-> winner: pow_lt
E(4333325939536502149, 6203777302392154398, 8398776702329832504, 4534766396126553545, 4207155762400068499, 1092200067244457229, 5745759890632121952) < 7129758684570177408 = False
	pow_lt_naive took 2.05E-4s
	pow_lt took 3.81E-6s
-> winner: pow_lt
E(1198688463183847310) < 7299516720137446658 = True
	pow_lt_naive took 1.91E-6s
	pow_lt took 1.43E-6s
-> winner: pow_lt
E(5915481424891413162, 5240418636774679484) < 8130862160215972019 = False
	pow_lt_naive took 1.09E-4s
	pow_lt took 3.58E-6s
-> winner: pow_lt

Statistics from 1000 test cases:
STATS 

----------------------------------------------------------------------
Ran 1 test in 0.145s

OK


### Results

The `pow_lt` implementation that checks the bit-length before computing the log and making a recursive call is much more efficient than `pow_lt_naive` that just uses logarithms.

## `pow_list`

`pow_list` takes as input a sequence of numbers $e_1,e_2,\ldots,e_\ell$ and returns the value of $E(e_1,e_2,\ldots,e_\ell)$.

In [10]:
def pow_list(seq: List[Decimal]) -> Decimal:
    """
    Given an arbitrarily long sequence of numbers seq = [e₁, e₂, e₃, ...],
    computes E(e₁, e₂, e₃, ...).

    The elements of seq are assumed to be numbers without validation.

    Parameters
    ----------
    seq: list of numbers.

    Returns
    -------
        E(seq[0], seq[1], seq[2], ...).
    """
    l = len(seq)
    if not l: # if len(seq) == 0
        return 1
    elif l == 1: # if len(seq) == 1
        return seq[0]

    def _pow_list(seq):
        """Auxiliary function."""
        if seq[0] == 1:
            return 1
        if len(seq) == 2:
            return seq[0]**seq[1]
        return seq[0]**_pow_list(seq[1:])
    
    return _pow_list(seq)

class TestPowList(BaseTest):
    def aux(self, num_cases: int, verb: BaseTest.V):
        custom = lambda seq: pow_list(seq)
        custom.__name__ = 'pow_list'
        cases = [([],)]
        for _ in range(num_cases-1):
            seq = list()
            for _ in range(1,4):
                seq.append(randrange(1,4))
            cases.append((seq,))
        self.compare_fns([custom], cases, verb=verb,
                         name_fn=lambda seq: 'E(%s)'%str(seq).strip('[]'))

    def test(self):
        super().test(n={self.CASES[0]: self.V.R, self.CASES[1]: self.V.SB})
        
if __name__ == '__main__':
    TestPowList().run_test()

E() = 1
	pow_list took 2.15E-6s
-> winner: pow_list
E(1, 2, 1) = 1
	pow_list took 2.62E-6s
-> winner: pow_list
E(3, 2, 2) = 81
	pow_list took 5.48E-6s
-> winner: pow_list

Statistics from 1000 test cases:
STATS | Function | Mean time
------|----------|-----------
#1.   | pow_list |  1.14E-6s



----------------------------------------------------------------------
Ran 1 test in 0.010s

OK


## The main function

`pow_mod` takes as input a sequence of positive integers $a_1,a_2,\ldots,a_\ell$ and a positive integer $m$ and returns $E(a_1,a_2,\ldots,a_\ell)\bmod m$.

In [12]:
def pow_mod(seq: List[int], m: int) -> int:
    """
    Given an arbitrarily long sequence of positive integers
    seq = [a₁, a₂, a₃, ...] and a positive integer m, computes
    E(a₁, a₂, a₃, ...) mod m.

    Parameters
    ----------
    seq: list of positive integers.
    m: positive integer
        The modulus of congruence.
    
    Returns
    -------
        E(seq[0], seq[1], seq[2], ...) mod m.
    """
    if m < 1:
        raise ValueError('m must be a positive integer')
    for a in seq:
        if a < 1:
            raise ValueError('seq may only contain positive integers')

    if m == 1: # 1 divides every integer
        return 0
    l = len(seq)
    if not l: # if len(seq) == 0
        return 1%m
    elif l == 1: # if len(seq) == 1
        return seq[0]%m

    def _pow_mod(seq, m):
        """Auxiliary function."""
        if m == 1: # 1 divides every integer
            return 0
        if len(seq) == 2: # recursive base case
            return pow(seq[0], seq[1], m)
        
        b, e = seq[0], seq[1:] # base and exponent
        g = gcd(b, m)
        if g == 1:
            return pow(b, _pow_mod(e, euler_totient(m)), m)
        
        n, k = m//g, 1
        g_ = gcd(g, n)
        while g_ > 1:
            n //= g_
            k += 1
            g_ = gcd(g, n)
        h = m//n
        _, x, y = ext_gcd(n, h)
        return (h*(y%n)*pow(b, _pow_mod(e, euler_totient(n)), n)+
                n*(x%h)*(pow(b, pow_list(e), h) if pow_lt(e, k) else 0))%m
    
    return _pow_mod(seq, m)

class TestPowMod(BaseTest):
    def val_test(self):
        cases = [([7**30*4850, 9878789, 1214712], 7**8*338),
                 ([101, 100, 99], 7),
                 ([2, 3, 4, 5], 100), ([3, 4, 5], 20),
                 ([6, 5, 4, 3, 2], 7**8*338)]
        val_set = [7**8*290, 4, 52, 1, 951546056]
        for i, (seq, m) in enumerate(cases):
            print('E(', end='')
            print(*seq, sep=', ', end='')
            print(') mod %i ='%m, end=' ')
            t0 = time()
            r = pow_mod(seq, m)
            t1 = time()
            self.assertEqual(r, val_set[i])
            print('%i\n\ttook %ss'%(r, self.fsci(t1-t0)))
    
    @staticmethod
    def dot_to_and(s: str, bold: bool = False) -> str:
        if bold:
            t = s.split('.')
            return '\\textbf{%s}&\\textbf{%s}'%(t[0],t[1]) if '.' in s else '\\textbf{%s}&'%s
        else:
            return s.replace('.','&') if '.' in s else s+'&'

    def benchmark(self, latex: bool = False, maxB: int = 10, maxL: int = 20):
        """
        Benchmark the runtime of pow_mod.
    
        Parameters
        ----------
        latex: bool (default is False)
            If set to False, prints out a table with the runtime data. If set to
            True, prints out the LaTeX code to generate the same table.
        maxB: int (default is 8)
            Tests integers from 3 bytes to maxB bytes.
        maxL: int (default is 20)
            Tests sequences of length 5 up to 5*maxL with a granularity of 5.
        """
        num_cases = 1000

        bs = [8*B for B in range(3,maxB+1)]
        ls = [5*l for l in range(1,maxL+1)]
        col1 = len(str(max(ls, key=lambda l: len(str(l)))))
        cols = max(len('9.99E-99'), len(str(max(bs, key=lambda b: len(str(b))))))
        times = [[None]*len(bs) for _ in range(len(ls))]

        if latex:
            print('\\begin{tabular}{ll|',*(['r@{.}l']*len(bs)), sep='', end='|r@{.}l}\n\t\\toprule\n')
            print('\t\\multicolumn{2}{c}{} & \\multicolumn{%i}{c}{max bit length $b$} \\tabularnewline'%(2*len(bs)))
            print('\t\\multicolumn{2}{c}{} ', *(['\\multicolumn{2}{c}{\\textbf{%i}} '%b for b in bs]), sep='& ', end=' & \\multicolumn{2}{c}{\\textbf{mean}} \\tabularnewline\n\t\\midrule\n')
        else:
            print(('%%%is'%col1)%' ', end='')
            for b in bs:
                print((' | %%%ii'%cols)%b, end='')
            print()
        for i,l in enumerate(ls): # sequence length
            if latex:
                if i:
                    print('\t', end='')
                else:
                    print('\t\\multirow{%i}{*}{\\shortstack{sequence\\\\length\\\\$\\ell$}}'%len(ls), end=' ')
                print('& \\multicolumn{1}{c|}{\\textbf{%i}}'%l, end='', flush=True)
            else:
                print(('%%%ii'%col1)%l, end='', flush=True)
            for j,b in enumerate(bs): # max bit length
                tot = 0
                for _ in range(num_cases):
                    seq = list()
                    for _ in range(l):
                        seq.append(randrange(1,2**b))
                    m = randint(1,2**b)
                    t0 = time()
                    r = pow_mod(seq, m)
                    t1 = time()
                    tot += t1-t0
                times[i][j] = 1000*tot/num_cases
                tstr = self.fpos(times[i][j])
                if latex:
                    print(' & %s'%self.dot_to_and(tstr), end='', flush=True)
                else:
                    print((' | %%%is'%cols)%tstr, end='', flush=True)
            if latex:
                print(' & %s \\tabularnewline'%self.dot_to_and(self.fpos(mean(times[i])),bold=False))
            if not latex:
                print()
        if latex:
            print('\t\\midrule\n\t& \\textbf{mean}', *(self.dot_to_and(self.fpos(mean([times[i][j] for i in range(len(ls))])),bold=False) for j in range(len(bs))), '\\multicolumn{1}{c}{}', sep=' & ', end='\\tabularnewline\n')
            print('\\bottomrule\n\\end{tabular}')
    
    def test_pow_mod(self):
        self.val_test()
        self.benchmark(latex=False)

if __name__ == '__main__':
    TextTestRunner(verbosity=0).run(TestSuite([TestPowMod('test_pow_mod')]))

E(109315800409857451726136757650, 9878789, 1214712) mod 1948502738 = 1671792290
	took 1.42E-4s
E(101, 100, 99) mod 7 = 4
	took 4.41E-5s
E(2, 3, 4, 5) mod 100 = 52
	took 6.29E-5s
E(3, 4, 5) mod 20 = 1
	took 4.48E-5s
E(6, 5, 4, 3, 2) mod 1948502738 = 951546056
	took 1.41E-4s
    |       24 |       32 |       40 |       48 |       56 |       64 |       72 |       80
  5 |     0.11 |     0.18 |     0.35 |     0.67 |      1.6 |     3.77 |    11.51 |    41.76
 10 |     0.15 |     0.25 |     0.43 |     0.76 |     1.61 |     4.26 |    17.41 |    33.98
 15 |     0.15 |     0.25 |     0.42 |     0.77 |     1.67 |     4.83 |    11.87 |    41.55
 20 |     0.15 |     0.23 |     0.45 |     0.76 |     1.64 |      4.6 |    11.31 |    37.96
 25 |     0.15 |     0.25 |     0.43 |     0.83 |     1.51 |     3.65 |     12.6 |    41.16
 30 |     0.15 |     0.26 |     0.41 |     0.78 |     1.63 |     4.85 |    12.72 |    40.94
 35 |     0.16 |     0.24 |     0.39 |     0.75 |      1.7 |     4.17 |    14.24 |

----------------------------------------------------------------------
Ran 1 test in 1174.141s

OK
